# CFN smoke test on `one_traj_loss_check.npy`

This notebook mirrors `tests/test_training.py::test_train_epoch_on_one_traj_loss_check`:

1. Load the Whitham trajectory `data/one_traj_loss_check.npy` with shape `(1, 3201, 500, 1)`
2. Sample training windows with `cfn.sampling.build_epoch_dataset`
3. Run one `train_epoch` draw on a small CFN (`dt=0.03125`, `dx=0.4`)

**Before running:** place the project on Colab using one of the setup options in the next cell.

## 1. Setup project + dependencies

Pick **one** of these options:

- **A. Git clone** — set `REPO_URL` below and run the cell.
- **B. Google Drive** — mount Drive, set `PROJECT_ROOT` to your cloned project folder.
- **C. Upload zip** — zip your local `cfn-project` folder, upload when prompted, then set `PROJECT_ROOT`.

You also need `one_traj_loss_check.npy` (gitignored locally). Upload it in the next section if it is not already under `data/`.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

# --- Option A: git clone (edit URL) ---
REPO_URL = None  # e.g. "https://github.com/<you>/cfn-project.git"

# --- Option B/C: existing folder on Colab or Drive ---
PROJECT_ROOT = Path("/content/cfn-project")  # change if your path differs

# Optional: mount Google Drive and point PROJECT_ROOT at your clone, e.g.
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/Modulation Project/cfn-project")

if REPO_URL:
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
elif not PROJECT_ROOT.exists():
    # Option C fallback: upload a zip of the project
    from google.colab import files

    print("Upload your cfn-project.zip (or change PROJECT_ROOT above).")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    subprocess.run(["unzip", "-q", zip_name, "-d", "/content"], check=True)
    if not PROJECT_ROOT.exists():
        candidates = [
            p for p in Path("/content").iterdir()
            if p.is_dir() and (p / "pyproject.toml").exists()
        ]
        if len(candidates) == 1:
            PROJECT_ROOT = candidates[0]
        else:
            raise FileNotFoundError(
                f"Could not find project at {PROJECT_ROOT}. "
                f"Set PROJECT_ROOT manually. Candidates: {candidates}"
            )

print(f"Project root: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "pyproject.toml").exists(), "pyproject.toml not found — check PROJECT_ROOT"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=PROJECT_ROOT, check=True)

import torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Load `one_traj_loss_check.npy`

Expected path: `data/one_traj_loss_check.npy`.

If the file is missing, run the upload cell below (or point `TRAJ_PATH` at a copy on Google Drive).

In [ ]:
import numpy as np
from pathlib import Path

TRAJ_PATH = PROJECT_ROOT / "data" / "one_traj_loss_check.npy"

# Optional: override with a Drive path, e.g.
# TRAJ_PATH = Path("/content/drive/MyDrive/Modulation Project/data/one_traj_loss_check.npy")

if not TRAJ_PATH.exists():
    from google.colab import files

    TRAJ_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f"Upload one_traj_loss_check.npy (will save to {TRAJ_PATH})")
    uploaded = files.upload()
    src = Path(next(iter(uploaded)))
    src.rename(TRAJ_PATH)

traj = np.load(TRAJ_PATH)
print("Path:", TRAJ_PATH)
print("Shape:", traj.shape)
print("dtype:", traj.dtype)
print("u range:", float(traj.min()), "->", float(traj.max()))
assert traj.shape == (1, 3201, 500, 1)

## 3. Run the smoke test

Same parameters as `configs/default.yaml`, scaled down for a quick Colab run.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from cfn import CFN
from cfn.sampling import build_epoch_dataset
from cfn.training import _make_fixed_validation_set, train_epoch

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- sampling check ---
rng = np.random.default_rng(0)
data = build_epoch_dataset(
    traj,
    L=160,
    window_t=700,
    window_x=80,
    num_samples=8,
    rng=rng,
    per_corner_fraction=0.05,
)
assert data["un"].shape == (8, 80, 1)
assert data["un_p1"].shape == (8, 4, 80, 1)
print("Sampling OK:", data["un"].shape, data["un_p1"].shape)

# --- one training epoch draw ---
model = CFN(features=[8, 8, 1], dt=0.03125, dx=0.4).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

val = _make_fixed_validation_set(
    traj,
    device=device,
    window_t=700,
    window_x=80,
    num_val_samples=8,
    L=160,
    val_per_corner_fraction=0.1,
)

train_loss, val_loss = train_epoch(
    model,
    traj,
    optimizer,
    loss_fn,
    device=device,
    fixed_val_data=val,
    window_t=700,
    window_x=80,
    num_samples=8,
    L=160,
    margin=16,
    rollout_steps=4,
    num_draws=1,
    train_per_corner_fraction=0.05,
)

assert np.isfinite(train_loss)
assert np.isfinite(val_loss)
print(f"PASS  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

## 4. Optional: quick trajectory preview

In [ ]:
import matplotlib.pyplot as plt

field = traj[0]  # [Nt, Nx, 1]
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(
    field[:, :, 0].T,
    aspect="auto",
    origin="lower",
    cmap="viridis",
)
ax.set_xlabel("time index")
ax.set_ylabel("space index")
ax.set_title("one_traj_loss_check.npy")
fig.colorbar(im, ax=ax, label="u")
plt.show()

## 5. Optional: run the pytest test locally in Colab

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_training.py::test_train_epoch_on_one_traj_loss_check", "-v"],
    cwd=PROJECT_ROOT,
    check=True,
)